# ECERAG round 2 — unconditional baseline, conflict quarantine, merged corrective retrieval

Builds on the round-1 results already committed in the repo (`results/kaggle/*/records.jsonl`);
only the new generations run here:

1. **Unconditional baseline** — same evidence, prompt without the "say NOT_SUPPORTED" instruction
   (reviewer 1: the original baseline refused via its prompt, confounding the gate's effect).
2. **Conflict quarantine** — when a conflict is detected, drop both conflicting passages and answer
   from the rest, instead of refusing every conflicted case.
3. **Merged corrective retrieval** — keep the original evidence, add re-retrieved passages, and accept
   the pass only if it scores clearly higher.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All**.
**Runtime:** about 3 h for all five models (~1.5 h if you keep only 7B and 14B). Resumable if the session dies.

In [ ]:
import os, socket, subprocess
try:
    socket.create_connection(("github.com", 443), timeout=5).close()
except OSError:
    raise SystemExit("No internet access: Settings -> Internet ON (phone-verified account), then restart and run all.")

REPO, BRANCH, WORKDIR = "https://github.com/yomisys/LLM", "kaggle-experiment", "/kaggle/working/LLM"
if os.path.isdir(WORKDIR):
    subprocess.run(["git", "-C", WORKDIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORKDIR], check=True)
os.chdir(WORKDIR)
print("repo ready at", os.getcwd())
!pip install -q rank_bm25 bitsandbytes

In [ ]:
GENERATORS = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct",
]
FOUR_BIT = {"Qwen/Qwen2.5-14B-Instruct"}
JUDGE_MODEL = "microsoft/Phi-3.5-mini-instruct"   # same judge as round 1
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
run_dir = lambda g: f"results/kaggle/{g.split('/')[-1]}"

### Smoke test — 3 records with the smallest model, into a scratch copy

In [ ]:
!mkdir -p /tmp/r2smoke && cp results/kaggle/Qwen2.5-0.5B-Instruct/run_config.json results/kaggle/Qwen2.5-0.5B-Instruct/records.jsonl /tmp/r2smoke/
!ECERAG_GENERATOR_MODEL=Qwen/Qwen2.5-0.5B-Instruct python code/augment_records.py /tmp/r2smoke --limit 3 && echo "smoke test OK"

### 1. Round-2 generations (GPU)

In [ ]:
for g in GENERATORS:
    print(f"===== {g}")
    q4 = "1" if g in FOUR_BIT else "0"
    !ECERAG_GENERATOR_MODEL={g} ECERAG_LOAD_4BIT={q4} python code/augment_records.py {run_dir(g)}

### 2. Judge the new answers (already-judged round-1 answers are skipped)

In [ ]:
for g in GENERATORS:
    !ECERAG_JUDGE_MODEL={JUDGE_MODEL} python code/judge_answers.py {run_dir(g)}

### 3. Analysis

In [ ]:
dirs = " ".join(run_dir(g) for g in GENERATORS)
!python code/analyze.py {dirs} --grader lexical > /dev/null
!python code/analyze.py {dirs} --grader judge > /dev/null
from IPython.display import Markdown, display
for g in GENERATORS:
    display(Markdown(open(f"{run_dir(g)}/report_judge.md").read()))

### 4. Download results

In [ ]:
!cd results && zip -qr /kaggle/working/ecerag_round2_results.zip kaggle && ls -lh /kaggle/working/ecerag_round2_results.zip